# Project: Customer Support with the Handoffs (State Machine) Pattern

This project builds a customer support agent whose behavior changes as it moves through a
workflow. Unlike Subagents, where a supervisor calls separate agents as tools, the **state
machine pattern** uses a **single agent** whose configuration (system prompt + tools) changes
dynamically based on where the conversation is.

The agent will:

- Collect warranty information before proceeding.
- Classify issues as hardware or software.
- Provide a solution, or escalate to a human.
- Maintain conversation state across multiple turns.

```mermaid
%%{init: {'theme':'base', 'themeVariables': {'primaryColor':'#E5F4FF','primaryTextColor':'#030710','primaryBorderColor':'#006DDD','lineColor':'#40668D','secondaryColor':'#F6FFDB','tertiaryColor':'#FDF3FF'}}}%%
flowchart TD
    Start([Customer reports an issue]) --> Warranty{Is the device under warranty?}
    Warranty -->|Yes| IssueType{What type of issue?}
    Warranty -->|No| OutOfWarranty{What type of issue?}
    IssueType -->|Hardware| Repair[Provide warranty repair instructions]
    IssueType -->|Software| Troubleshoot[Provide troubleshooting steps]
    OutOfWarranty -->|Hardware| Escalate[Escalate to human for paid repair]
    OutOfWarranty -->|Software| Troubleshoot
    Troubleshoot --> Close([Issue resolved])
    Repair --> Close
    Escalate --> Close

    classDef startEnd fill:#F6FFDB,stroke:#6E8900,stroke-width:2px,color:#2E3900
    classDef decisionNode fill:#E5F4FF,stroke:#006DDD,stroke-width:2px,color:#030710
    classDef actionNode fill:#FDF3FF,stroke:#7E65AE,stroke-width:2px,color:#504B5F
    classDef escalateNode fill:#F8E8E6,stroke:#B27D75,stroke-width:2px,color:#634643
    class Start,Close startEnd
    class Warranty,IssueType,OutOfWarranty decisionNode
    class Repair,Troubleshoot actionNode
    class Escalate escalateNode
```

Each "step" above is just a different configuration of the *same underlying agent*, selected
dynamically based on state -- not a separate agent instance.

## Setup

Copy `.env.example` to `.env` and set `OPENAI_API_KEY=sk-...` before running this notebook.


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is not set. Copy .env.example to .env and add your key.")

from langchain.chat_models import init_chat_model

model = init_chat_model("openai:gpt-5-mini")
print("Model ready.")


## 1. Define custom state

The `current_step` field is the core of the state machine: it determines which prompt and
tools are loaded on each turn.


In [ ]:
from langchain.agents import AgentState
from typing_extensions import NotRequired
from typing import Literal

SupportStep = Literal["warranty_collector", "issue_classifier", "resolution_specialist"]

class SupportState(AgentState):
    """State for the customer support workflow."""
    current_step: NotRequired[SupportStep]
    warranty_status: NotRequired[Literal["in_warranty", "out_of_warranty"]]
    issue_type: NotRequired[Literal["hardware", "software"]]

print("State schema defined.")


## 2. Create tools that manage workflow state

The key idea: tools return `Command` objects that update **both** the data they collected
*and* `current_step` -- which is what actually drives the workflow forward.

```mermaid
graph LR
    T1[record_warranty_status] -->|Command update| S1[warranty_status +<br/>current_step = issue_classifier]
    T2[record_issue_type] -->|Command update| S2[issue_type +<br/>current_step = resolution_specialist]
```


In [ ]:
from langchain.tools import tool, ToolRuntime
from langchain.messages import ToolMessage
from langgraph.types import Command

@tool
def record_warranty_status(
    status: Literal["in_warranty", "out_of_warranty"],
    runtime: ToolRuntime[None, SupportState],
) -> Command:
    """Record the customer's warranty status and transition to issue classification."""
    return Command(
        update={
            "messages": [
                ToolMessage(content=f"Warranty status recorded as: {status}", tool_call_id=runtime.tool_call_id)
            ],
            "warranty_status": status,
            "current_step": "issue_classifier",
        }
    )


@tool
def record_issue_type(
    issue_type: Literal["hardware", "software"],
    runtime: ToolRuntime[None, SupportState],
) -> Command:
    """Record the type of issue and transition to resolution specialist."""
    return Command(
        update={
            "messages": [
                ToolMessage(content=f"Issue type recorded as: {issue_type}", tool_call_id=runtime.tool_call_id)
            ],
            "issue_type": issue_type,
            "current_step": "resolution_specialist",
        }
    )


@tool
def escalate_to_human(reason: str) -> str:
    """Escalate the case to a human support specialist."""
    return f"Escalating to human support. Reason: {reason}"


@tool
def provide_solution(solution: str) -> str:
    """Provide a solution to the customer's issue."""
    return f"Solution provided: {solution}"

print("Workflow tools defined.")


## 3. Define each step's prompt and tools

A simple dictionary maps each step name to its prompt, its tools, and what state it requires
before it can run.


In [ ]:
WARRANTY_COLLECTOR_PROMPT = """You are a customer support agent helping with device issues.

CURRENT STAGE: Warranty verification

At this step, you need to:
1. Greet the customer warmly
2. Ask if their device is under warranty
3. Use record_warranty_status to record their response and move to the next step

Be conversational and friendly. Don't ask multiple questions at once."""

ISSUE_CLASSIFIER_PROMPT = """You are a customer support agent helping with device issues.

CURRENT STAGE: Issue classification
CUSTOMER INFO: Warranty status is {warranty_status}

At this step, you need to:
1. Ask the customer to describe their issue
2. Determine if it's a hardware issue (physical damage, broken parts) or software issue (app crashes, performance)
3. Use record_issue_type to record the classification and move to the next step

If unclear, ask clarifying questions before classifying."""

RESOLUTION_SPECIALIST_PROMPT = """You are a customer support agent helping with device issues.

CURRENT STAGE: Resolution
CUSTOMER INFO: Warranty status is {warranty_status}, issue type is {issue_type}

At this step, you need to:
1. For SOFTWARE issues: provide troubleshooting steps using provide_solution
2. For HARDWARE issues:
   - If IN WARRANTY: explain warranty repair process using provide_solution
   - If OUT OF WARRANTY: escalate_to_human for paid repair options

Be specific and helpful in your solutions."""

STEP_CONFIG = {
    "warranty_collector": {
        "prompt": WARRANTY_COLLECTOR_PROMPT,
        "tools": [record_warranty_status],
        "requires": [],
    },
    "issue_classifier": {
        "prompt": ISSUE_CLASSIFIER_PROMPT,
        "tools": [record_issue_type],
        "requires": ["warranty_status"],
    },
    "resolution_specialist": {
        "prompt": RESOLUTION_SPECIALIST_PROMPT,
        "tools": [provide_solution, escalate_to_human],
        "requires": ["warranty_status", "issue_type"],
    },
}
print("Step configuration defined -- add a new step just by adding a new dictionary entry.")


## 4. Create step-based middleware

Middleware reads `current_step` from state on every model call, looks up the matching
configuration, and overrides the prompt and tools for that turn only.


In [ ]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def apply_step_config(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    """Configure agent behavior based on the current step."""
    current_step = request.state.get("current_step", "warranty_collector")
    step_config = STEP_CONFIG[current_step]

    for key in step_config["requires"]:
        if request.state.get(key) is None:
            raise ValueError(f"{key} must be set before reaching {current_step}")

    system_prompt = step_config["prompt"].format(**request.state)
    request = request.override(system_prompt=system_prompt, tools=step_config["tools"])
    return handler(request)

print("Middleware defined.")


## 5. Create the agent

A checkpointer is required here too -- without it, `current_step` would be lost between
messages, and the workflow would restart from scratch on every turn.


In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

all_tools = [record_warranty_status, record_issue_type, provide_solution, escalate_to_human]

# Keep a reference to the checkpointer itself, separate from the agent -- section 7 below
# rebuilds the agent with an expanded toolset, and reusing this same object (instead of a
# fresh InMemorySaver()) is what keeps the conversation history intact across that rebuild.
checkpointer = InMemorySaver()

support_agent = create_agent(
    model,
    tools=all_tools,
    state_schema=SupportState,
    middleware=[apply_step_config],
    checkpointer=checkpointer,
)
print("Support agent created.")


## 6. Walk through the workflow, turn by turn

```mermaid
sequenceDiagram
    participant User
    participant Agent as Agent (same instance)
    User->>Agent: "My phone screen is cracked"
    Note over Agent: current_step = warranty_collector (default)
    Agent->>User: "Sorry to hear that! Is it under warranty?"
    User->>Agent: "Yes, still under warranty"
    Agent->>Agent: record_warranty_status -> current_step = issue_classifier
    Agent->>User: "What's the issue -- hardware or software?"
    User->>Agent: "Screen physically cracked from dropping it"
    Agent->>Agent: record_issue_type -> current_step = resolution_specialist
    Agent->>User: "Here's the warranty repair process..."
```


In [ ]:
from langchain.messages import HumanMessage
from langchain_core.utils.uuid import uuid7

thread_id = str(uuid7())
config = {"configurable": {"thread_id": thread_id}}

print("=== Turn 1: Warranty Collection ===")
result = support_agent.invoke({"messages": [HumanMessage("Hi, my phone screen is cracked")]}, config)
for msg in result["messages"]:
    msg.pretty_print()


In [ ]:
print("=== Turn 2: Warranty Response ===")
result = support_agent.invoke({"messages": [HumanMessage("Yes, it's still under warranty")]}, config)
for msg in result["messages"]:
    msg.pretty_print()
print(f"\nCurrent step: {result.get('current_step')}")


In [ ]:
print("=== Turn 3: Issue Description ===")
result = support_agent.invoke({"messages": [HumanMessage("The screen is physically cracked from dropping it")]}, config)
for msg in result["messages"]:
    msg.pretty_print()
print(f"\nCurrent step: {result.get('current_step')}")


In [ ]:
print("=== Turn 4: Resolution ===")
result = support_agent.invoke({"messages": [HumanMessage("What should I do?")]}, config)
for msg in result["messages"]:
    msg.pretty_print()


## 7. Add flexibility: letting the customer go back

Not every workflow needs this, and not every transition makes sense (you typically can't go
back once a refund has been processed) -- but for this one, let's allow correcting the
warranty status or issue classification from the resolution step.

<p></p>

> If your workflow needs arbitrary transitions between most steps, consider whether a
> structured state machine is the right shape at all -- this pattern works best with a clear
> sequential progression plus occasional backward corrections, not a free-for-all.

> **A real gotcha worth knowing:** it's tempting to just extend `STEP_CONFIG["resolution_specialist"]["tools"]`
> with the new tools and keep using the existing agent. In the currently installed LangChain
> version, that raises a clear error instead: *"Middleware added tools that the agent doesn't
> know how to execute... This happens when middleware modifies `request.tools` in
> `wrap_model_call` to include tools that weren't passed to `create_agent()`."* Tools have to be
> registered at agent construction time -- middleware can only narrow which of those registered
> tools are visible per step, not introduce new ones. So below we rebuild `all_tools` to include
> the two new tools and recreate the agent with them -- but we reuse the **same** `checkpointer`
> object from section 5, not a new one, so the conversation history survives the rebuild.

> **A second gotcha:** every tool call the model makes needs a matching `ToolMessage` back in
> the state update, the same way `record_warranty_status` and `record_issue_type` already return
> one above. A tool that only returns `Command(update={"current_step": ...})` with no
> `ToolMessage` raises *"Expected to have a matching ToolMessage in Command.update... Every tool
> call... MUST have a corresponding ToolMessage"* the first time it's actually called. So
> `go_back_to_warranty` and `go_back_to_classification` need a `ToolRuntime` parameter and a
> `ToolMessage` in their update, just like the other state-changing tools.


In [ ]:
@tool
def go_back_to_warranty(runtime: ToolRuntime[None, SupportState]) -> Command:
    """Go back to warranty verification step."""
    return Command(
        update={
            "messages": [ToolMessage(content="Going back to warranty verification.", tool_call_id=runtime.tool_call_id)],
            "current_step": "warranty_collector",
        }
    )


@tool
def go_back_to_classification(runtime: ToolRuntime[None, SupportState]) -> Command:
    """Go back to issue classification step."""
    return Command(
        update={
            "messages": [ToolMessage(content="Going back to issue classification.", tool_call_id=runtime.tool_call_id)],
            "current_step": "issue_classifier",
        }
    )


STEP_CONFIG["resolution_specialist"]["tools"].extend([go_back_to_warranty, go_back_to_classification])

RESOLUTION_SPECIALIST_PROMPT = """You are a customer support agent helping with device issues.

CURRENT STAGE: Resolution
CUSTOMER INFO: Warranty status is {warranty_status}, issue type is {issue_type}

At this step, you need to:
1. For SOFTWARE issues: provide troubleshooting steps using provide_solution
2. For HARDWARE issues:
   - If IN WARRANTY: explain warranty repair process using provide_solution
   - If OUT OF WARRANTY: escalate_to_human for paid repair options

If the customer indicates any information was wrong, use:
- go_back_to_warranty to correct warranty status
- go_back_to_classification to correct issue type

Be specific and helpful in your solutions."""
STEP_CONFIG["resolution_specialist"]["prompt"] = RESOLUTION_SPECIALIST_PROMPT

all_tools = all_tools + [go_back_to_warranty, go_back_to_classification]
support_agent = create_agent(
    model,
    tools=all_tools,
    state_schema=SupportState,
    middleware=[apply_step_config],
    checkpointer=checkpointer,  # the SAME checkpointer from section 5 -- not a new one
)

result = support_agent.invoke({"messages": [HumanMessage("Actually, I made a mistake - my device is out of warranty")]}, config)
for msg in result["messages"]:
    msg.pretty_print()


## 8. Managing message history as the workflow grows

As the agent progresses through steps, the message list keeps growing. `SummarizationMiddleware`
compresses earlier messages once a token threshold is crossed, while keeping recent context:

```python
from langchain.agents.middleware import SummarizationMiddleware

support_agent = create_agent(
    model,
    tools=all_tools,
    state_schema=SupportState,
    middleware=[
        apply_step_config,
        SummarizationMiddleware(model="gpt-5-mini", trigger=("tokens", 4000), keep=("messages", 10)),
    ],
    checkpointer=InMemorySaver(),
)
```

This is shown here for reference rather than run live, since it only matters once a
conversation has grown large enough to need it -- see the short-term memory guide for the full
set of memory-management options.

## Understanding state transitions

| Turn | `current_step` going in | What happens | `current_step` after |
|---|---|---|---|
| 1 | `warranty_collector` (default) | Agent asks about warranty | `warranty_collector` (unchanged until tool call) |
| 2 | `warranty_collector` | `record_warranty_status` called | `issue_classifier` |
| 3 | `issue_classifier` | `record_issue_type` called | `resolution_specialist` |
| 4 | `resolution_specialist` | `provide_solution` or `escalate_to_human` called | `resolution_specialist` (terminal) |

**The key insight: tools drive the workflow by updating `current_step`, and middleware
responds by applying the right configuration on the next turn.**

## Key takeaways

| Point | Why it matters |
|---|---|
| One agent, many configurations | Simpler than spinning up a separate agent per step |
| `current_step` lives in state, not in code | The same middleware function handles every step |
| Tools return `Command` to drive transitions | State changes and workflow progression happen together |
| A checkpointer is mandatory | Without it, `current_step` resets every turn |
| `requires` validation catches a skipped step early | Fails loudly instead of silently misbehaving |

**When to use this pattern:** a task that flows through clear, mostly-sequential stages, where
behavior should change as the task progresses, and where the agent needs to converse directly
with the user at each stage. For centralized orchestration across unrelated domains instead,
see the Subagents notebook.
